In [ ]:
import itertools
import typing
from pathlib import Path

import pandas as pd

from utils.plot_corr_assoc_utils import create_corr_assoc_matrix
from utils.utils import load_data

# Hardcoded paths, change to apply to new data
data_path = Path("data/preprocessed/esa_2018_2021.pkl")
vars_info_path = Path("data/preprocessed/esa_vars_info.pkl")

# Read in data
data: pd.DataFrame = load_data(data_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

### search for variable information

In [ ]:
vars_info[vars_info["name"].str.contains("hnb")]

### count values of variable

In [ ]:
data["hnb_fag_sch"].value_counts(dropna=False)

### show how often two variables are equal and different

In [ ]:
# Check if two variables are equal (only works if both are continuous or categorical with the same labels)
vari, varj = "hnb30", "typ_hnb"

# True means vari is equal to varj, False means they are different
vars_equal = (data[vari] == data[varj]) | (data[vari].isnull() & data[varj].isnull())
vars_equal.value_counts(normalize=True)  # normalize=False for absolute values, normalize=True for percentages

In [ ]:
# Show instances where they are not equal
data.loc[~vars_equal, (vari, varj)]

In [ ]:
# Check if two categorical variables are the same
vari, varj = "alkkons", "alkrisk"
# Use normalize="index" to show percentages for each index
pd.crosstab(data[vari], data[varj], dropna=False, normalize=False)

### create table of highest correlations

In [ ]:
# Get variable groups
var_groups = {group: vars_info.loc[vars_info["group"] == group, "name"] for group in vars_info["group"].cat.categories}
var_groups = typing.cast(dict[str, list[str]], var_groups)

# Visualize correlation matrices
treat_cat_as_continuous: list[str] = vars_info.loc[vars_info["type"].isin(["binary", "ordinal"]), "name"].tolist()

corr_assoc_matrix, p_value_matrix = create_corr_assoc_matrix(data, treat_cat_as_continuous)
table = (
    pd.DataFrame(
        [
            [vari, varj, corr_assoc, p_value_matrix.loc[vari, varj]]
            for vari, varj in itertools.combinations(vars_info["name"], r=2)
            if (corr_assoc := corr_assoc_matrix.loc[vari, varj]) > 0.9
        ]
    )
    .set_axis(["vari", "varj", "corr_assoc", "p_value"], axis=1)
    .sort_values("corr_assoc", ascending=False)
)
table